<a href="https://colab.research.google.com/github/srinivas340/ml_practice/blob/main/2_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's begin by importing the (torch.nn) package from Pytorch, which contains utility classes for building neural networks.

In [401]:
import torch.nn as nn
import torch

In [402]:
import numpy as np

#INput(temp, rainfall, humidity)
inputs = np.array([[73,67,43],[91,88,64],[87,134,58],
                   [102,43,37],[69,96,70],[73,67,43],
                   [69,96,70],[73,67,43],[91,88,64],
                  [87,134,58],[102,43,37],[69,96,70]],dtype='float32')

#Targets (apples, oranges)
targets = np.array([[56,70],[81,101],[119,133],
                    [22,37],[103,119],[56,70],
                    [81,101],[119,133],[22,37],
                    [103,119],[56,70],[81,101]],
                    dtype='float32')

**DAtaset and DataLoader**

we'll creat a (TensorDataset), which allows access to rows from inputs and targets as tuples, and provides standard API's for working with many different types of datasets in pytorch

In [403]:
from torch.utils.data import TensorDataset

In [404]:
# Define dataset
if not isinstance(inputs, torch.Tensor):
    inputs = torch.from_numpy(inputs)
if not isinstance(targets, torch.Tensor):
    targets = torch.from_numpy(targets)
train_ds = TensorDataset(inputs, targets)
train_ds[0:3]

(tensor([[ 73.,  67.,  43.],
         [ 91.,  88.,  64.],
         [ 87., 134.,  58.]]),
 tensor([[ 56.,  70.],
         [ 81., 101.],
         [119., 133.]]))

In [405]:
from torch.utils.data import DataLoader

In [406]:
#define data Loader
batch_size=5
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [407]:
for xb, yb in train_dl:
  print(xb)
  print(yb)
  break;

tensor([[102.,  43.,  37.],
        [ 87., 134.,  58.],
        [ 73.,  67.,  43.],
        [ 87., 134.,  58.],
        [ 91.,  88.,  64.]])
tensor([[ 56.,  70.],
        [103., 119.],
        [ 56.,  70.],
        [119., 133.],
        [ 81., 101.]])


**nn.linear**


insted of initializing the weights and biasses manually, we can define the model using the **nn.Linear** class from pytorch,which does it automatically

In [408]:
#Define model
model=nn.Linear(3,2)
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[ 0.5679, -0.4434,  0.2313],
        [ 0.3037,  0.1603, -0.2634]], requires_grad=True)
Parameter containing:
tensor([0.4304, 0.5476], requires_grad=True)


pytorch models also have a helpful **.parameters** method,which returns a list containing all the weights and bias matrices presented in the model.

for our linear regression model, we have one weight matrix and one bias matrix.

In [409]:
#parameters
list(model.parameters())

[Parameter containing:
 tensor([[ 0.5679, -0.4434,  0.2313],
         [ 0.3037,  0.1603, -0.2634]], requires_grad=True),
 Parameter containing:
 tensor([0.4304, 0.5476], requires_grad=True)]

we can use the model to generate predictions in the exact same way as before:

In [410]:
#generate parameters
preds = model(inputs)
preds

tensor([[22.1277, 22.1262],
        [27.8965, 25.4259],
        [ 3.8415, 33.1639],
        [47.8505, 28.6669],
        [13.2432, 18.4466],
        [22.1277, 22.1262],
        [13.2432, 18.4466],
        [22.1277, 22.1262],
        [27.8965, 25.4259],
        [ 3.8415, 33.1639],
        [47.8505, 28.6669],
        [13.2432, 18.4466]], grad_fn=<AddmmBackward0>)

**LOss funnction**

instead of defining a loss function manually, we can use the built-in loss function  **mse_loss**.

In [411]:
#import nn.functional
import torch.nn.functional as F


**nn.functional** package contains many useful loss functions and several other utilities.

In [412]:
#define loss function
loss_fn=F.mse_loss

loss for the current predictions of our model

In [413]:
loss = loss_fn(model(inputs),targets)
print(loss)

tensor(5049.0317, grad_fn=<MseLossBackward0>)


#Optimizer


INstead of manually manipulating the model's weights & biases using gradients,

 we can use the optimizer  (**optim.SGD**)

 SGD stands for stochastic greadient descent .

  it is called stochastic .because samples are selected in batches(often with random shuffling) inste of as a single group.

In [414]:
#Define optimizer
opt = torch.optim.SGD(model.parameters(),lr=1e-5)

Note that ***model.parameters()\*** is passed as an argument to ***optim.SGD***,

 so that the optimizer knows which matrix should be modified during the update step.

 Also we can specify a learning rate which controls the amount by which the parameters are modified.

# Train the model


we are now ready to train the model. we'll follow the exact same process to implement gradient descent:

1.Generate predictions .

2.calculate the loss

3.compare greadients w.r.t the weights and biases

4.Adjust the weights by subtracting a small quantity proportional to the gradient

5.reset the greadients to the zero





The only change that we'll work batches of data, instead of processing the entire traing data in every iteration. let's define a utility function ***fit***

which trains the model for a given number of epochs.

In [415]:
# Utility function to train the model
def fit(num_epochs,model,loss_fn,opt):
  #Repeat for given number of epochs
  for epoch in range(num_epochs):

    #Train with batches of data
    for xb,yb in train_dl:

      #1. Generate predictions
      pred = model(xb)

      #2. Calculate loss
      loss = loss_fn(pred,yb)

      #3. compute gradients
      loss.backward()

      #4. Update parameters using gradients
      opt.step()

      #5. Reset the gradients to zero
      opt.zero_grad()

  #Print the progress
  if((epoch+1)%10==0):
    print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1,num_epochs,loss.item()))



In [416]:
fit(200,model,loss_fn,opt)

Epoch [200/200], Loss: 16.3559


In [417]:
#generate predictions of our model
preds=model(inputs)
preds

tensor([[ 58.9582,  73.3118],
        [ 79.4838,  96.2136],
        [108.6645, 133.4776],
        [ 41.2739,  57.0758],
        [ 86.4511, 100.1351],
        [ 58.9582,  73.3118],
        [ 86.4511, 100.1351],
        [ 58.9582,  73.3118],
        [ 79.4838,  96.2136],
        [108.6645, 133.4776],
        [ 41.2739,  57.0758],
        [ 86.4511, 100.1351]], grad_fn=<AddmmBackward0>)

In [418]:
#compare with targets
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.]])